<a href="https://colab.research.google.com/github/liillikoi/Splicing-Guided-Genomic-FHE/blob/main/benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install tenseal cryptography numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 35.0 MB/s  0:00:00


In [9]:
import gzip
import time
import os
import hashlib
import hmac
import numpy as np
import tenseal as ts
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ==========================================
# 0. 루트 경로(/) 기준으로 파일 자동 탐색
# ==========================================
VCF_FILE = "/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz" if os.path.exists("/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz") else "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz"
GTF_FILE = "/gencode.v45.annotation.gtf.gz" if os.path.exists("/gencode.v45.annotation.gtf.gz") else "gencode.v45.annotation.gtf.gz"

VCF_FILE = f"/{VCF_NAME}" if os.path.exists(f"/{VCF_NAME}") else VCF_NAME
GTF_FILE = f"/{GTF_NAME}" if os.path.exists(f"/{GTF_NAME}") else GTF_NAME

print(f"연결된 VCF 경로: {VCF_FILE}")
print(f"연결된 GTF 경로: {GTF_FILE}\n")

# ==========================================
# 1. GTF 파일에서 22번 염색체 Exon 구간(좌표) 추출
# ==========================================
print("1. GTF 파일에서 Exon 좌표 로딩 중...")
exon_intervals = []
with gzip.open(GTF_FILE, 'rt', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 5:
            continue
        chrom = parts[0].replace('chr', '')
        feature = parts[2]
        if chrom == '22' and feature == 'exon':
            start, end = int(parts[3]), int(parts[4])
            exon_intervals.append((start, end))

exon_intervals.sort()
merged_exons = []
for start, end in exon_intervals:
    if not merged_exons or merged_exons[-1][1] < start:
        merged_exons.append([start, end])
    else:
        merged_exons[-1][1] = max(merged_exons[-1][1], end)

def is_in_exon(pos, intervals):
    low, high = 0, len(intervals) - 1
    while low <= high:
        mid = (low + high) // 2
        if intervals[mid][0] <= pos <= intervals[mid][1]:
            return True
        elif pos < intervals[mid][0]:
            high = mid - 1
        else:
            low = mid + 1
    return False

print(f"   -> 병합된 Exon 구간 개수: {len(merged_exons)}개\n")

# ==========================================
# 2. VCF 파일 파싱 및 Exon/Intron 스플라이싱
# ==========================================
print("2. VCF 파일에서 환자 변이 데이터 추출 및 Splicing 수행 중...")
exon_variants = []
intron_variants = []

max_lines = 50000
count = 0

with gzip.open(VCF_FILE, 'rt', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 10:
            continue
        pos = int(parts[1])
        gt_str = parts[9].split(':')[0]
        genotype = sum(int(a) for a in gt_str.replace('/', '|').split('|') if a.isdigit())

        if is_in_exon(pos, merged_exons):
            exon_variants.append(float(genotype))
        else:
            intron_variants.append(f"{pos}:{genotype}")

        count += 1
        if max_lines and count >= max_lines:
            break

print(f"   -> 분석 변이 총합: {count}개 | Exon 변이: {len(exon_variants)}개 | Intron 변이: {len(intron_variants)}개\n")

# ==========================================
# 3. 더미 패딩 및 비밀키 기반 슬롯 셔플링
# ==========================================
BUCKET_SIZE = 8192
SECRET_SEED = 20261113

padded_exon = np.zeros(BUCKET_SIZE, dtype=np.float64)
actual_len = min(len(exon_variants), BUCKET_SIZE)
padded_exon[:actual_len] = exon_variants[:actual_len]

rng = np.random.default_rng(SECRET_SEED)
perm_indices = rng.permutation(BUCKET_SIZE)
shuffled_exon = padded_exon[perm_indices]

# ==========================================
# 4. 하이브리드 암호화 (Exon: CKKS / Intron: AES-256)
# ==========================================
print("3. 하이브리드 암호화 시작...")
start_enc = time.time()

context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=16384,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2**40

enc_exon_vector = ts.ckks_vector(context, shuffled_exon)
serialized_enc_exon = enc_exon_vector.serialize()

aes_key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)
intron_bytes = ",".join(intron_variants).encode('utf-8')
enc_intron_bytes = aesgcm.encrypt(nonce, intron_bytes, None)

hmac_key = os.urandom(32)
integrity_tag = hmac.new(hmac_key, serialized_enc_exon + enc_intron_bytes, hashlib.sha256).hexdigest()

end_enc = time.time()
print(f"   -> 암호화 완료 시간: {end_enc - start_enc:.4f}초")
print(f"   -> Exon FHE 암호문 크기: {len(serialized_enc_exon) / (1024*1024):.2f} MB")
print(f"   -> Intron AES 암호문 크기: {len(enc_intron_bytes) / 1024:.2f} KB")
print(f"   -> 무결성 검증 태그(HMAC): {integrity_tag[:16]}...\n")

# ==========================================
# 5. 클라우드 동형암호 연산 (PRS 계산)
# ==========================================
print("4. 클라우드 서버에서 복호화 없이 PRS(위험점수) 연산 중...")
raw_weights = np.zeros(BUCKET_SIZE, dtype=np.float64)
raw_weights[:actual_len] = np.random.uniform(0.1, 0.5, size=actual_len)
shuffled_weights = raw_weights[perm_indices]

start_eval = time.time()
enc_prs_result = enc_exon_vector.dot(shuffled_weights)
end_eval = time.time()

dec_prs = enc_prs_result.decrypt()[0]
plain_prs = np.dot(padded_exon, raw_weights)

print(f"   -> 클라우드 연산 시간: {end_eval - start_eval:.4f}초")
print(f"   -> 평문 계산 PRS 결과: {plain_prs:.6f}")
print(f"   -> 동형암호 복호화 결과: {dec_prs:.6f}")
print(f"   -> 절대 오차: {abs(plain_prs - dec_prs):.8f}")


연결된 VCF 경로: /ALL.chr22.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz
연결된 GTF 경로: /gencode.v45.annotation.gtf.gz

1. GTF 파일에서 Exon 좌표 로딩 중...
   -> 병합된 Exon 구간 개수: 6798개

2. VCF 파일에서 환자 변이 데이터 추출 및 Splicing 수행 중...
   -> 분석 변이 총합: 50000개 | Exon 변이: 5119개 | Intron 변이: 44881개

3. 하이브리드 암호화 시작...
   -> 암호화 완료 시간: 0.3878초
   -> Exon FHE 암호문 크기: 0.63 MB
   -> Intron AES 암호문 크기: 482.13 KB
   -> 무결성 검증 태그(HMAC): 32c5e09c12a60e0b...

4. 클라우드 서버에서 복호화 없이 PRS(위험점수) 연산 중...
   -> 클라우드 연산 시간: 0.0461초
   -> 평문 계산 PRS 결과: 142.747423
   -> 동형암호 복호화 결과: 142.747625
   -> 절대 오차: 0.00020119


In [8]:
import gzip
import math
import time
import numpy as np
import tenseal as ts

# ==========================================
# 1. 22번 염색체 전체 변이 데이터 로드
# ==========================================
print("22번 염색체 전체 변이 로딩 중 (약 10~15초 소요)...")
all_variants = []
exon_variants = []
intron_variants = []

with gzip.open(VCF_FILE, 'rt', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.strip().split('\t')
        if len(parts) < 10:
            continue
        pos = int(parts[1])
        gt_str = parts[9].split(':')[0]
        genotype = float(sum(int(a) for a in gt_str.replace('/', '|').split('|') if a.isdigit()))

        all_variants.append(genotype)
        if is_in_exon(pos, merged_exons):
            exon_variants.append(genotype)
        else:
            intron_variants.append(f"{pos}:{int(genotype)}")

total_cnt = len(all_variants)
exon_cnt = len(exon_variants)
print(f"-> 전체 변이 수: {total_cnt:,}개 | Exon 변이 수: {exon_cnt:,}개 ({exon_cnt/total_cnt*100:.2f}%)\n")

# ==========================================
# 2. 대조군 (Baseline): 전체 변이 일괄 동형암호화 및 연산
# ==========================================
print("[대조군] Baseline (전체 변이 일괄 FHE) 측정 중...")
num_chunks_base = math.ceil(total_cnt / BUCKET_SIZE)
base_weights = np.random.uniform(0.1, 0.5, size=total_cnt)

t0 = time.time()
base_ciphertexts = []
base_size_bytes = 0
for i in range(num_chunks_base):
    chunk = np.zeros(BUCKET_SIZE, dtype=np.float64)
    sub = all_variants[i*BUCKET_SIZE : (i+1)*BUCKET_SIZE]
    chunk[:len(sub)] = sub
    ct = ts.ckks_vector(context, chunk)
    base_ciphertexts.append(ct)
    base_size_bytes += len(ct.serialize())
base_enc_time = time.time() - t0

t0 = time.time()
base_res = None
for i, ct in enumerate(base_ciphertexts):
    w_chunk = np.zeros(BUCKET_SIZE, dtype=np.float64)
    sub_w = base_weights[i*BUCKET_SIZE : (i+1)*BUCKET_SIZE]
    w_chunk[:len(sub_w)] = sub_w
    dot_ct = ct.dot(w_chunk)
    base_res = dot_ct if base_res is None else base_res + dot_ct
base_eval_time = time.time() - t0

# ==========================================
# 3. 실험군 (Proposed): 스플라이싱 기반 하이브리드 FHE
# ==========================================
print("[실험군] Proposed (Splicing FHE + AES) 측정 중...")
num_chunks_prop = math.ceil(exon_cnt / BUCKET_SIZE)
prop_weights = np.random.uniform(0.1, 0.5, size=exon_cnt)

t0 = time.time()
prop_ciphertexts = []
prop_fhe_bytes = 0
for i in range(num_chunks_prop):
    chunk = np.zeros(BUCKET_SIZE, dtype=np.float64)
    sub = exon_variants[i*BUCKET_SIZE : (i+1)*BUCKET_SIZE]
    chunk[:len(sub)] = sub
    shuffled_chunk = chunk[perm_indices]
    ct = ts.ckks_vector(context, shuffled_chunk)
    prop_ciphertexts.append(ct)
    prop_fhe_bytes += len(ct.serialize())

intron_raw = ",".join(intron_variants).encode('utf-8')
enc_intron = aesgcm.encrypt(nonce, intron_raw, None)
prop_total_bytes = prop_fhe_bytes + len(enc_intron)
prop_enc_time = time.time() - t0

t0 = time.time()
prop_res = None
for i, ct in enumerate(prop_ciphertexts):
    w_chunk = np.zeros(BUCKET_SIZE, dtype=np.float64)
    sub_w = prop_weights[i*BUCKET_SIZE : (i+1)*BUCKET_SIZE]
    w_chunk[:len(sub_w)] = sub_w
    shuffled_w = w_chunk[perm_indices]
    dot_ct = ct.dot(shuffled_w)
    prop_res = dot_ct if prop_res is None else prop_res + dot_ct
prop_eval_time = time.time() - t0

# ==========================================
# 4. 최종 논문용 벤치마크 비교 리포트 출력
# ==========================================
print("\n" + "="*65)
print(f"{'평가 지표 (Metric)':<25} | {'Baseline (전체 FHE)':<18} | {'Proposed (스플라이싱)':<18}")
print("="*65)
print(f"{'FHE 암호문 개수':<25} | {num_chunks_base:>15} 개 | {num_chunks_prop:>15} 개")
print(f"{'총 암호문 용량 (MB)':<25} | {base_size_bytes/(1024*1024):>14.2f} MB | {prop_total_bytes/(1024*1024):>14.2f} MB")
print(f"{'클라이언트 암호화 시간':<25} | {base_enc_time:>14.4f} 초 | {prop_enc_time:>14.4f} 초")
print(f"{'클라우드 동형 연산 시간':<25} | {base_eval_time:>14.4f} 초 | {prop_eval_time:>14.4f} 초")
print("="*65)
print(f"★ 용량 절감률: {(1 - prop_total_bytes/base_size_bytes)*100:.2f}% 감소")
print(f"★ 연산 속도 향상: {base_eval_time/prop_eval_time:.2f}배 빨라짐")


22번 염색체 전체 변이 로딩 중 (약 10~15초 소요)...
-> 전체 변이 수: 1,103,547개 | Exon 변이 수: 107,814개 (9.77%)

[대조군] Baseline (전체 변이 일괄 FHE) 측정 중...
[실험군] Proposed (Splicing FHE + AES) 측정 중...

평가 지표 (Metric)            | Baseline (전체 FHE)  | Proposed (스플라이싱)  
FHE 암호문 개수                |             135 개 |              14 개
총 암호문 용량 (MB)             |          84.70 MB |          19.23 MB
클라이언트 암호화 시간              |         1.9260 초 |         0.2258 초
클라우드 동형 연산 시간             |         6.8476 초 |         0.6499 초
★ 용량 절감률: 77.30% 감소
★ 연산 속도 향상: 10.54배 빨라짐
